# Simulator method metrics (breast-level)

Collects per-method, per-site **and** pooled breast-level metrics from the NVFLARE-simulator
prediction CSVs.

- **Unit:** breast (mean prob / max label per (exam, laterality)) via `fl_utils.breast_aggregate`.
- **Source:** the per-round `incoming_global` prediction CSVs (received-aggregate eval each round; for **FedBN** this is the global backbone + that site's local BN — i.e. the per-site FedBN model).
- **Round selection:** (A) each site's own best-val-AUC round; (B) one common round maximizing **pooled** val AUC.
- **Operating point:** Youden's J on validation → applied to test (per-site for site rows; pooled-val for the Pooled row).
- **CIs:** AUC → DeLong **and** bootstrap; sensitivity/specificity → bootstrap.

Edit the **Config** cell (paths, methods) then **Run All**. Writes two CSVs + a markdown table to `OUT_DIR`.

Covers **FedProx** & **FedBN** now (uncomment **FedAvg** when it finishes — same trajectory). Ditto / module-wise report the *personal* model, whose per-round preds aren't dumped (only the final round), so handle those separately.

In [10]:
# ---- Config: edit paths / methods here, then Run All ----
import os, re, glob, csv, json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import norm
from IPython.display import display

# Reuse the codebase's EXACT breast aggregation (mean prob, max label per (exam, laterality)).
# Point at any sim job's app/custom (identical across jobs); adjust to your checkout.
FL_UTILS_DIR = "/raid/home/lsollis/GMIC/GMIC/gmic_job_fedbn_sim/app/custom"

CLIENTS = ["UHCC", "HPU", "RSNA-GCP"]
METHODS = {
    "FedProx (mu=0.1)": "/raid/home/lsollis/GMIC/GMIC/sim/fedprox_mu0.1/{site}",
    "FedBN":            "/raid/home/lsollis/GMIC/GMIC/sim/fedbn/{site}",
    # Add when finished (same incoming_global trajectory):
    # "FedAvg":         "/raid/home/lsollis/GMIC/GMIC/sim/fedavg/fedavg/{site}",
}
TAG     = "incoming_global"   # per-round received-aggregate eval; FedBN = global backbone + that site's local BN
N_BOOT  = 2000
SEED    = 0
CI      = 0.95
OUT_DIR = "/raid/home/lsollis/GMIC/GMIC"

In [5]:
# ---- Helpers: aggregation, AUC + DeLong CI, Youden threshold, point metrics, bootstrap ----
sys.path.insert(0, FL_UTILS_DIR)
from fl_utils import breast_aggregate   # canonical view -> breast aggregation

def safe_auc(labels, scores):
    labels = np.asarray(labels)
    return float(roc_auc_score(labels, scores)) if len(np.unique(labels)) > 1 else float("nan")

def delong_ci(labels, scores, alpha=CI):
    """AUC + DeLong CI for a single classifier (placement-value form; exact, O(m*n))."""
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    pos = scores[labels == 1]; neg = scores[labels == 0]
    m, n = len(pos), len(neg)
    if m == 0 or n == 0:
        return float("nan"), float("nan"), float("nan")
    cmp = (pos[:, None] > neg[None, :]).astype(float) + 0.5 * (pos[:, None] == neg[None, :])
    auc = float(cmp.mean())
    if m < 2 or n < 2:
        return auc, float("nan"), float("nan")
    V10 = cmp.mean(axis=1); V01 = cmp.mean(axis=0)
    se = float(np.sqrt(V10.var(ddof=1) / m + V01.var(ddof=1) / n))
    z = float(norm.ppf(1 - (1 - alpha) / 2))
    return auc, max(0.0, auc - z * se), min(1.0, auc + z * se)

def youden_threshold(labels, scores):
    fpr, tpr, thr = roc_curve(labels, scores)
    return float(thr[int(np.argmax(tpr - fpr))])

def point_metrics(labels, scores, thr):
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    pred = (scores >= thr).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum()); tn = int(((pred == 0) & (labels == 0)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum()); fn = int(((pred == 0) & (labels == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return sens, spec

def bootstrap_ci(labels, scores, thr, n_boot=N_BOOT, seed=SEED, alpha=CI):
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    rng = np.random.default_rng(seed); idx = np.arange(len(labels))
    aucs, senss, specs = [], [], []
    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        yb, sb = labels[b], scores[b]
        if len(np.unique(yb)) < 2:
            continue
        aucs.append(roc_auc_score(yb, sb))
        s, p = point_metrics(yb, sb, thr); senss.append(s); specs.append(p)
    def q(a):
        if not a:
            return (float("nan"), float("nan"))
        return (float(np.nanpercentile(a, 100 * (1 - alpha) / 2)),
                float(np.nanpercentile(a, 100 * (1 + alpha) / 2)))
    return q(aucs), q(senss), q(specs)

def load_breast(method_tmpl, site, split, tag=TAG):
    """round -> (breast_probs, breast_labels) from the per-view prediction CSVs."""
    d = method_tmpl.format(site=site)
    out = {}
    for path in glob.glob(os.path.join(d, f"{site}_predictions_{tag}_round*_{split}.csv")):
        mo = re.search(r"_round(\d+)_", os.path.basename(path))
        if not mo:
            continue
        with open(path, newline="") as f:
            rows = list(csv.DictReader(f))
        if not rows:
            continue
        bp, by = breast_aggregate([r["exam_id"] for r in rows], [r["view"] for r in rows],
                                  [float(r["prob_malignant"]) for r in rows], [int(r["label"]) for r in rows])
        out[int(mo.group(1))] = (np.asarray(bp, float), np.asarray(by, int))
    return out

In [8]:
# ---- Build the two tables: (A) per-site best-val round, (B) pooled best round ----
def _row(method, site, rnd, by_test, bp_test, thr):
    auc, dlo, dhi = delong_ci(by_test, bp_test)
    (alo, ahi), (slo, shi), (plo, phi) = bootstrap_ci(by_test, bp_test, thr)
    sens, spec = point_metrics(by_test, bp_test, thr)
    ci = lambda lo, hi: f"[{lo:.3f}, {hi:.3f}]"
    return {"method": method, "site": site, "round": int(rnd),
            "n_breasts": int(len(by_test)), "prevalence": round(float(np.mean(by_test)), 4),
            "AUC": round(auc, 4), "AUC_DeLong95": ci(dlo, dhi), "AUC_boot95": ci(alo, ahi),
            "threshold": round(float(thr), 4),
            "sensitivity": round(sens, 4), "sens_boot95": ci(slo, shi),
            "specificity": round(spec, 4), "spec_boot95": ci(plo, phi)}

rows_persite, rows_pooled = [], []
for name, tmpl in METHODS.items():
    val  = {s: load_breast(tmpl, s, "val")  for s in CLIENTS}
    test = {s: load_breast(tmpl, s, "test") for s in CLIENTS}
    if any(not val[s] or not test[s] for s in CLIENTS):
        print(f"[skip] {name}: missing CSVs ->")
        for s in CLIENTS:
            print(f"    {s}: {len(val[s])} val / {len(test[s])} test rounds  in {tmpl.format(site=s)}")
        continue

    # (A) per-site best-val round: each site picks its own round by its val AUC
    for s in CLIENTS:
        r = max(val[s], key=lambda rr: safe_auc(val[s][rr][1], val[s][rr][0]))
        thr = youden_threshold(val[s][r][1], val[s][r][0])
        rows_persite.append(_row(name, s, r, test[s][r][1], test[s][r][0], thr))

    # (B) pooled best round: one round (present at all sites) maximizing POOLED val AUC
    common = sorted(set.intersection(*[set(val[s]) & set(test[s]) for s in CLIENTS]))
    def pooled_val_auc(r):
        by = np.concatenate([val[s][r][1] for s in CLIENTS]); bp = np.concatenate([val[s][r][0] for s in CLIENTS])
        return safe_auc(by, bp)
    rp = max(common, key=pooled_val_auc)
    for s in CLIENTS:
        thr = youden_threshold(val[s][rp][1], val[s][rp][0])
        rows_pooled.append(_row(name, s, rp, test[s][rp][1], test[s][rp][0], thr))
    by_v = np.concatenate([val[s][rp][1] for s in CLIENTS]); bp_v = np.concatenate([val[s][rp][0] for s in CLIENTS])
    by_t = np.concatenate([test[s][rp][1] for s in CLIENTS]); bp_t = np.concatenate([test[s][rp][0] for s in CLIENTS])
    rows_pooled.append(_row(name, "Pooled", rp, by_t, bp_t, youden_threshold(by_v, bp_v)))

df_persite = pd.DataFrame(rows_persite)
df_pooled  = pd.DataFrame(rows_pooled)

In [11]:
# ---- Display + export for the paper agent ----
print("=== (A) PER-SITE best-val-round selection (each site at its own best val round) ===")
display(df_persite)
print("\n=== (B) POOLED best-round selection (common round maximizing pooled val AUC) ===")
display(df_pooled)

os.makedirs(OUT_DIR, exist_ok=True)
df_persite.to_csv(os.path.join(OUT_DIR, "metrics_persite_bestval.csv"), index=False)
df_pooled.to_csv(os.path.join(OUT_DIR, "metrics_pooled_bestround.csv"), index=False)
try:
    md = ("# Simulator metrics (breast-level, Youden-on-val operating point)\n\n"
          "## (A) Per-site best-val-round\n\n" + df_persite.to_markdown(index=False) +
          "\n\n## (B) Pooled best-round\n\n" + df_pooled.to_markdown(index=False) + "\n")
    with open(os.path.join(OUT_DIR, "metrics_for_paper.md"), "w") as f:
        f.write(md)
    print("\nwrote metrics_for_paper.md (+ 2 CSVs) to", OUT_DIR)
except Exception as e:
    print("markdown export skipped (pip install tabulate for .md):", e)
    print("CSVs written to", OUT_DIR)

=== (A) PER-SITE best-val-round selection (each site at its own best val round) ===


,method,site,round,n_breasts,prevalence,AUC,AUC_DeLong95,AUC_boot95,threshold,sensitivity,sens_boot95,specificity,spec_boot95
0,FedProx (mu=0.1),UHCC,42,878,0.1185,0.9077,"[0.873, 0.943]","[0.871, 0.939]",0.5685,0.7404,"[0.653, 0.820]",0.9147,"[0.894, 0.934]"
1,FedProx (mu=0.1),HPU,2,374,0.1283,0.8581,"[0.790, 0.926]","[0.786, 0.922]",0.4101,0.7083,"[0.575, 0.837]",0.8742,"[0.837, 0.909]"
2,FedProx (mu=0.1),RSNA-GCP,0,396,0.1237,0.8411,"[0.766, 0.917]","[0.766, 0.917]",0.0704,0.7551,"[0.633, 0.881]",0.8415,"[0.802, 0.880]"
3,FedBN,UHCC,37,878,0.1185,0.8997,"[0.862, 0.937]","[0.859, 0.935]",0.5436,0.7212,"[0.634, 0.808]",0.9406,"[0.923, 0.957]"
4,FedBN,HPU,1,374,0.1283,0.8302,"[0.757, 0.903]","[0.751, 0.903]",0.5253,0.5833,"[0.442, 0.727]",0.9172,"[0.886, 0.945]"
5,FedBN,RSNA-GCP,0,396,0.1237,0.8411,"[0.766, 0.917]","[0.766, 0.917]",0.0704,0.7551,"[0.633, 0.881]",0.8415,"[0.802, 0.880]"



=== (B) POOLED best-round selection (common round maximizing pooled val AUC) ===


,method,site,round,n_breasts,prevalence,AUC,AUC_DeLong95,AUC_boot95,threshold,sensitivity,sens_boot95,specificity,spec_boot95
0,FedProx (mu=0.1),UHCC,4,878,0.1185,0.8758,"[0.835, 0.916]","[0.835, 0.914]",0.5431,0.7596,"[0.678, 0.842]",0.8695,"[0.845, 0.894]"
1,FedProx (mu=0.1),HPU,4,374,0.1283,0.8510,"[0.779, 0.923]","[0.773, 0.920]",0.5846,0.6250,"[0.488, 0.761]",0.9601,"[0.937, 0.979]"
2,FedProx (mu=0.1),RSNA-GCP,4,396,0.1237,0.8252,"[0.750, 0.900]","[0.746, 0.900]",0.4676,0.7959,"[0.679, 0.909]",0.7061,"[0.656, 0.753]"
3,FedProx (mu=0.1),Pooled,4,1648,0.1220,0.8572,"[0.825, 0.889]","[0.825, 0.888]",0.4805,0.7662,"[0.706, 0.825]",0.8106,"[0.791, 0.831]"
4,FedBN,UHCC,5,878,0.1185,0.8930,"[0.856, 0.930]","[0.855, 0.928]",0.6008,0.6250,"[0.536, 0.718]",0.9509,"[0.936, 0.966]"
5,FedBN,HPU,5,374,0.1283,0.8221,"[0.744, 0.900]","[0.738, 0.897]",0.6029,0.5417,"[0.395, 0.688]",0.9417,"[0.915, 0.966]"
6,FedBN,RSNA-GCP,5,396,0.1237,0.8326,"[0.758, 0.908]","[0.755, 0.907]",0.4584,0.7755,"[0.655, 0.891]",0.7406,"[0.695, 0.786]"
7,FedBN,Pooled,5,1648,0.1220,0.8594,"[0.827, 0.892]","[0.827, 0.892]",0.4759,0.7562,"[0.695, 0.814]",0.8286,"[0.810, 0.848]"


markdown export skipped (pip install tabulate for .md): `Import tabulate` failed.  Use pip or conda to install the tabulate package.
CSVs written to /raid/home/lsollis/GMIC/GMIC
